In [1]:
import tensorflow as tf

I0000 00:00:1777857568.386889    5005 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777857568.448886    5005 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777857569.487780    5005 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:

# Check for GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Prevent TF from pre-allocating all 100% of VRAM
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✅ GPU Detected: {gpus}")
    except RuntimeError as e:
        print(e)
else:
    print("❌ GPU not detected. Script will run on CPU.")

✅ GPU Detected: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
import gc
from numba import cuda

# 1. Delete large objects (like your model or heavy arrays)
# del model 
# del embeddings

# 2. Force Python's Garbage Collector
gc.collect()

# 3. Reset the TensorFlow/Keras Backend
tf.keras.backend.clear_session()

In [4]:
import os
import cv2
import numpy as np
from deepface import DeepFace
import shutil

In [6]:
dataset_root = "/mnt/d/aa-kuliah/Skripsi/Dataset/first-impressions"

In [26]:
train_path = os.path.join(dataset_root, "train")
num_frames_to_extract = 30 
passed_videos = 0

for file_name in os.listdir(train_path):
   if passed_videos >= 1300:
      print("✅ Sudah mencapai 1300 video yang lolos. Proses selesai.")
      break

   video_path = os.path.join(train_path, file_name)
   cap = cv2.VideoCapture(video_path)
   
   total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
   interval = max(1, total_frames // num_frames_to_extract)
   
   file_base_name = os.path.splitext(file_name)[0]
   img_out_dir = os.path.join("datasets/extracted/ImageData/trainingData", file_base_name)
   feat_out_dir = os.path.join("datasets/extracted/Features/visual/trainingData", file_base_name)

   os.makedirs(img_out_dir, exist_ok=True)
   os.makedirs(feat_out_dir, exist_ok=True)

   extracted_count = 0
   fail_count = 0
   
   last_feat_data = None  # To store the numpy array
   last_img_data = None   # To store the image with border
   
   # We will iterate exactly 30 times based on calculated intervals
   for i in range(num_frames_to_extract):
      frame_idx = i * interval
      cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
      ret, frame = cap.read()
      
      if not ret:
         break

      frame_resized = cv2.resize(frame, (480, 240))
      success = False

      try:
         # Extraction
         results = DeepFace.represent(
            img_path=frame_resized,
            model_name="VGG-Face",
            detector_backend="opencv", # Faster GPU performance
            enforce_detection=True
         )
         
         # If successful, extract data
         vector = results[0]["embedding"]
         face = results[0]["facial_area"]
         
         # Create the annotated image
         annotated_img = frame_resized.copy()
         cv2.rectangle(annotated_img, (face['x'], face['y']), 
                        (face['x']+face['w'], face['y']+face['h']), (0, 255, 0), 2)
         
         # Update "Last Success" storage
         last_feat_data = vector
         last_img_data = annotated_img
         success = True

      except Exception:
         fail_count += 1
         # Logic: If fail at index 0, we simply don't have a 'last_success' yet.
         # The next iteration (i+1) will try to find a face to start the sequence.
         if last_feat_data is not None:
            # Use fallback from previous frame
            vector = last_feat_data
            annotated_img = last_img_data
            success = True
         else:
            print(f"Index {i} failed and no previous frame available. Searching next...")

      if success:
         # Save Feature
         np.save(os.path.join(feat_out_dir, f"feature{extracted_count}.npy"), vector)
         # Save Image
         cv2.imwrite(os.path.join(img_out_dir, f"frame{extracted_count}.jpg"), annotated_img)
         extracted_count += 1

      # Check Fail Rate (strictly > 8)
      if fail_count > 8:
         print(f"Fail rate too high (>8) for {file_name}. Deleting folders.")
         break

   cap.release()

   # FINAL VALIDATION
   # If we didn't get 30 frames OR fail rate exceeded 8
   if extracted_count < num_frames_to_extract or fail_count > 8:
      if os.path.exists(img_out_dir): shutil.rmtree(img_out_dir)
      if os.path.exists(feat_out_dir): shutil.rmtree(feat_out_dir)
      print(f"Skipped video: {file_name}")
   else:
      passed_videos += 1
      print(f"Success: {file_name} | Passed: {passed_videos}")

Success: --Ymqszjv54.001.mp4 | Passed: 1
Success: --Ymqszjv54.003.mp4 | Passed: 2
Index 0 failed and no previous frame available. Searching next...
Skipped video: --Ymqszjv54.004.mp4
Index 0 failed and no previous frame available. Searching next...
Index 1 failed and no previous frame available. Searching next...
Skipped video: --Ymqszjv54.005.mp4
Success: -2qsCrkXdWs.001.mp4 | Passed: 3
Success: -55DRRMTppE.000.mp4 | Passed: 4
Success: -55DRRMTppE.005.mp4 | Passed: 5
Success: -5riMLK-PgU.001.mp4 | Passed: 6
Success: -6otZ7M-Mro.000.mp4 | Passed: 7
Success: -6otZ7M-Mro.001.mp4 | Passed: 8
Success: -8asrRvfJWA.003.mp4 | Passed: 9
Success: -8asrRvfJWA.004.mp4 | Passed: 10
Success: -9BZ8A9U7TE.000.mp4 | Passed: 11
Success: -9BZ8A9U7TE.003.mp4 | Passed: 12
Success: -9BZ8A9U7TE.004.mp4 | Passed: 13
Success: -agCXYgb7pI.004.mp4 | Passed: 14
Success: -AmMDnVl4s8.003.mp4 | Passed: 15
Success: -aU2FN5pkWA.001.mp4 | Passed: 16
Success: -aU2FN5pkWA.004.mp4 | Passed: 17
Success: -awAu11kBZ0.000.mp

In [ ]:
train_feature_root = os.path.join("datasets/extracted/Features/visual/trainingData")
train_image_root = os.path.join("datasets/extracted/ImageData/trainingData")

training_features = os.listdir(train_feature_root)
training_features = {folder: len(os.listdir(os.path.join(train_feature_root, folder))) for folder in training_features}

less_30_features_val = {k: v for k, v in training_features.items() if v < 30}

for folder_name in less_30_features_val.keys():
	feature_folder = os.path.join(train_feature_root, folder_name)
	image_folder = os.path.join(train_image_root, folder_name)

	if os.path.exists(feature_folder):
		shutil.rmtree(feature_folder)
		print(f"Removed feature folder: {feature_folder}")

	if os.path.exists(image_folder):
		shutil.rmtree(image_folder)
		print(f"Removed image folder: {image_folder}")

In [ ]:
# take only 1200 videos
train_feature_lists = os.listdir(train_feature_root)
train_feature_files = sorted(train_feature_lists)[:1200]

train_disposal_files = [f for f in train_feature_lists if f not in train_feature_files]

for folder_name in train_disposal_files:
	feature_folder = os.path.join(train_feature_root, folder_name)
	image_folder = os.path.join(train_image_root, folder_name)

	if os.path.exists(feature_folder):
		shutil.rmtree(feature_folder)
		print(f"Removed feature folder: {feature_folder}")

	if os.path.exists(image_folder):
		shutil.rmtree(image_folder)
		print(f"Removed image folder: {image_folder}")

[]

In [8]:
val_path = os.path.join(dataset_root, "val")
num_frames_to_extract = 30 
passed_videos = 0

for file_name in os.listdir(val_path):
   if passed_videos >= 400:
      print("✅ Sudah mencapai 400 video yang lolos. Proses selesai.")
      break

   video_path = os.path.join(val_path, file_name)
   cap = cv2.VideoCapture(video_path)
   
   total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
   interval = max(1, total_frames // num_frames_to_extract)
   
   file_base_name = os.path.splitext(file_name)[0]
   img_out_dir = os.path.join("datasets/extracted/ImageData/validationData", file_base_name)
   feat_out_dir = os.path.join("datasets/extracted/Features/visual/validationData", file_base_name)

   os.makedirs(img_out_dir, exist_ok=True)
   os.makedirs(feat_out_dir, exist_ok=True)

   extracted_count = 0
   fail_count = 0
   
   last_feat_data = None  # To store the numpy array
   last_img_data = None   # To store the image with border
   
   # We will iterate exactly 30 times based on calculated intervals
   for i in range(num_frames_to_extract):
      frame_idx = i * interval
      cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
      ret, frame = cap.read()
      
      if not ret:
         break

      frame_resized = cv2.resize(frame, (480, 240))
      success = False

      try:
         # Extraction
         results = DeepFace.represent(
            img_path=frame_resized,
            model_name="VGG-Face",
            detector_backend="opencv", # Faster GPU performance
            enforce_detection=True
         )
         
         # If successful, extract data
         vector = results[0]["embedding"]
         face = results[0]["facial_area"]
         
         # Create the annotated image
         annotated_img = frame_resized.copy()
         cv2.rectangle(annotated_img, (face['x'], face['y']), 
                        (face['x']+face['w'], face['y']+face['h']), (0, 255, 0), 2)
         
         # Update "Last Success" storage
         last_feat_data = vector
         last_img_data = annotated_img
         success = True

      except Exception:
         fail_count += 1
         # Logic: If fail at index 0, we simply don't have a 'last_success' yet.
         # The next iteration (i+1) will try to find a face to start the sequence.
         if last_feat_data is not None:
            # Use fallback from previous frame
            vector = last_feat_data
            annotated_img = last_img_data
            success = True
         else:
            print(f"Index {i} failed and no previous frame available. Searching next...")

      if success:
         # Save Feature
         np.save(os.path.join(feat_out_dir, f"feature{extracted_count}.npy"), vector)
         # Save Image
         cv2.imwrite(os.path.join(img_out_dir, f"frame{extracted_count}.jpg"), annotated_img)
         extracted_count += 1

      # Check Fail Rate (strictly > 8)
      if fail_count > 8:
         print(f"Fail rate too high (>8) for {file_name}. Deleting folders.")
         break

   cap.release()

   # FINAL VALIDATION
   # If we didn't get 30 frames OR fail rate exceeded 8
   if extracted_count < num_frames_to_extract or fail_count > 8:
      if os.path.exists(img_out_dir): shutil.rmtree(img_out_dir)
      if os.path.exists(feat_out_dir): shutil.rmtree(feat_out_dir)
      print(f"Skipped video: {file_name}")
   else:
      passed_videos += 1
      print(f"Success: {file_name} | Passed: {passed_videos}")

I0000 00:00:1777857681.623076    5005 cuda_dnn.cc:461] Loaded cuDNN version 91900
W0000 00:00:1777857682.572174    5005 bfc_allocator.cc:383] Garbage collection: deallocate free memory regions (i.e., allocations) so that we can re-allocate a larger region to avoid OOM due to memory fragmentation. If you see this message frequently, you are running near the threshold of the available device memory and re-allocation may incur great performance overhead. You may try smaller batch sizes to observe the performance impact. Set TF_ENABLE_GPU_GARBAGE_COLLECTION=false if you'd like to disable this feature.


Success: -6otZ7M-Mro.003.mp4 | Passed: 1
Success: -6otZ7M-Mro.005.mp4 | Passed: 2
Success: -8asrRvfJWA.001.mp4 | Passed: 3
Success: -9BZ8A9U7TE.002.mp4 | Passed: 4


I0000 00:00:1777857696.002182    6391 service.cc:153] XLA service 0x790afc043800 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777857696.002209    6391 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9 (Driver: 13.2.0; Runtime: 12.5.0; Toolkit: 12.5.0; DNN: 9.19.0)
I0000 00:00:1777857696.028780    6391 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
W0000 00:00:1777857697.633402    7938 hlo_rematerialization.cc:3204] Can't reduce memory use below 4.19GiB (4502794772 bytes) by rematerialization; only reduced to 4.88GiB (5245272096 bytes), down from 4.88GiB (5245272096 bytes) originally
W0000 00:00:1777857707.989079    6391 bfc_allocator.cc:502] Allocator (GPU_0_bfc) ran out of memory trying to allocate 4.88GiB (rounded to 5245239808)requested by op 
If the cause is memory fragmentation maybe the environment variable 'TF_GP

Success: -agCXYgb7pI.000.mp4 | Passed: 5
Success: -agCXYgb7pI.001.mp4 | Passed: 6
Success: -AmMDnVl4s8.001.mp4 | Passed: 7
Success: -BDSWZIF-WY.000.mp4 | Passed: 8
Success: -DOqN0d8KHw.000.mp4 | Passed: 9
Success: -DOqN0d8KHw.001.mp4 | Passed: 10
Success: -fqiCqZtgYs.000.mp4 | Passed: 11
Success: -N6QKrbnaDs.002.mp4 | Passed: 12
Success: -PF2TG2loGc.002.mp4 | Passed: 13
Success: -PWjgx2czwY.000.mp4 | Passed: 14
Index 0 failed and no previous frame available. Searching next...
Skipped video: -R2SZu3SYgM.002.mp4
Success: -R2SZu3SYgM.003.mp4 | Passed: 15
Success: -RhPxhfpg4U.004.mp4 | Passed: 16
Success: -RqxrwIxMvE.002.mp4 | Passed: 17
Success: -VTqcHNgH7M.001.mp4 | Passed: 18
Success: -VTqcHNgH7M.005.mp4 | Passed: 19
Success: -wO0oBNlEvY.005.mp4 | Passed: 20
Success: -Wqk9eex6bQ.001.mp4 | Passed: 21
Success: -zNyDPzId4E.000.mp4 | Passed: 22
Success: 04oq2yrBwMg.000.mp4 | Passed: 23
Success: 04oq2yrBwMg.003.mp4 | Passed: 24
Success: 04oq2yrBwMg.005.mp4 | Passed: 25
Success: 05l5bteT_qA.0

W0000 00:00:1777858254.208478   16122 hlo_rematerialization.cc:3204] Can't reduce memory use below 4.19GiB (4502700257 bytes) by rematerialization; only reduced to 4.89GiB (5246468128 bytes), down from 4.89GiB (5246468128 bytes) originally
W0000 00:00:1777858264.536107    6388 bfc_allocator.cc:502] Allocator (GPU_0_bfc) ran out of memory trying to allocate 4.89GiB (rounded to 5246419456)requested by op 
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
I0000 00:00:1777858264.536210    6388 bfc_allocator.cc:1049] BFCAllocator dump for GPU_0_bfc
I0000 00:00:1777858264.536213    6388 bfc_allocator.cc:1056] Bin (256): 	Total Chunks: 31, Chunks in use: 30. 7.8KiB allocated for chunks. 7.5KiB in use in bin. 676B client-requested in use in bin.
I0000 00:00:1777858264.536217    6388 bfc_allocator.cc:1056] Bin (512): 	Total Chunks: 2, Chunks

Success: 4lIbWq27O84.005.mp4 | Passed: 158
Success: 4lj66h4CXI8.001.mp4 | Passed: 159
Success: 4lj66h4CXI8.004.mp4 | Passed: 160
Success: 4nir0YtjHVs.003.mp4 | Passed: 161
Success: 4nir0YtjHVs.004.mp4 | Passed: 162
Success: 4nMstrTuXXw.000.mp4 | Passed: 163
Success: 4NzqG0XwOWg.003.mp4 | Passed: 164
Success: 4OKYywIqnqg.004.mp4 | Passed: 165
Success: 4Qr1BjZhnpg.005.mp4 | Passed: 166
Success: 4sCa35TOxgs.003.mp4 | Passed: 167
Success: 4T9UUWWRE94.001.mp4 | Passed: 168
Success: 4vJ69g7gAH4.000.mp4 | Passed: 169
Success: 4vJ69g7gAH4.003.mp4 | Passed: 170
Success: 4vJ69g7gAH4.005.mp4 | Passed: 171
Success: 4VwcU0ROy5k.000.mp4 | Passed: 172
Success: 4VwcU0ROy5k.005.mp4 | Passed: 173
Success: 4XdZDodpzac.001.mp4 | Passed: 174
Success: 4XdZDodpzac.002.mp4 | Passed: 175
Success: 4yogPbHFQ9o.000.mp4 | Passed: 176
Index 0 failed and no previous frame available. Searching next...
Skipped video: 4ZlcaXadwlo.002.mp4
Success: 4ZlcaXadwlo.004.mp4 | Passed: 177
Success: 5-mwIFjOWZ4.002.mp4 | Passed: 

KeyboardInterrupt: 

In [37]:
val_feature_root = os.path.join("datasets/extracted/Features/visual/validationData")
val_image_root = os.path.join("datasets/extracted/ImageData/validationData")

training_features = os.listdir(val_feature_root)
training_features = {folder: len(os.listdir(os.path.join(val_feature_root, folder))) for folder in training_features}

less_30_features_val = {k: v for k, v in training_features.items() if v < 30}

for folder_name in less_30_features_val.keys():
	feature_folder = os.path.join(val_feature_root, folder_name)
	image_folder = os.path.join(val_image_root, folder_name)

	if os.path.exists(feature_folder):
		shutil.rmtree(feature_folder)
		print(f"Removed feature folder: {feature_folder}")

	if os.path.exists(image_folder):
		shutil.rmtree(image_folder)
		print(f"Removed image folder: {image_folder}")

In [40]:
# take only 400 videos
val_feature_lists = os.listdir(val_feature_root)
val_feature_files = sorted(val_feature_lists)[:400]

val_disposal_files = [f for f in val_feature_lists if f not in val_feature_files]

# val_disposal_files

for folder_name in val_disposal_files:
	feature_folder = os.path.join(val_feature_root, folder_name)
	image_folder = os.path.join(val_image_root, folder_name)

	if os.path.exists(feature_folder):
		shutil.rmtree(feature_folder)
		print(f"Removed feature folder: {feature_folder}")

	if os.path.exists(image_folder):
		shutil.rmtree(image_folder)
		print(f"Removed image folder: {image_folder}")

Removed feature folder: datasets/extracted/Features/visual/validationData/cdtZKwM5NIM.003
Removed image folder: datasets/extracted/ImageData/validationData/cdtZKwM5NIM.003
Removed feature folder: datasets/extracted/Features/visual/validationData/b2Y41GrCrPs.004
Removed image folder: datasets/extracted/ImageData/validationData/b2Y41GrCrPs.004
Removed feature folder: datasets/extracted/Features/visual/validationData/bYXRyimxh7A.005
Removed image folder: datasets/extracted/ImageData/validationData/bYXRyimxh7A.005
Removed feature folder: datasets/extracted/Features/visual/validationData/clMg72MI8rw.000
Removed image folder: datasets/extracted/ImageData/validationData/clMg72MI8rw.000
Removed feature folder: datasets/extracted/Features/visual/validationData/bcRPLKygrNk.005
Removed image folder: datasets/extracted/ImageData/validationData/bcRPLKygrNk.005
Removed feature folder: datasets/extracted/Features/visual/validationData/c4XnKouozXU.005
Removed image folder: datasets/extracted/ImageData